[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/trigonometry/t06_multiple_angles.ipynb)

# A 45th-degree equation, solved with trigonometry: double and triple angles

In 1593 Adriaan van Roomen challenged the world's mathematicians with an equation of degree 45. The ambassador told
Henri IV no Frenchman could solve it, so the king gave it to François Viète. Viète saw that it was sin 45θ written in
terms of sin θ. He found a root almost at once, and soon all 23 positive ones.

The idea behind it: **turning the arrow twice, or three times, or 45 times, is still just a turn.**

**How to use this notebook.** Press **Run all** (or click a grey box and press Shift + Enter, box by box, from the
top). Then drag the sliders and watch. Change any number you like and run the box again. Nothing can break: if
something looks odd, run everything again from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this series of notebooks. Run this box first.
import math
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[CHALK, TEAL, SAGE, LILAC, CORAL]),
})

def circle_axes(ax, lim=1.35, labels=True):
    """The unit circle with light axes: the stage for the turning arrow."""
    t = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(t), np.sin(t), color=DIM, lw=1.6)
    ax.axhline(0, color=DIM, lw=0.9); ax.axvline(0, color=DIM, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.grid(False)
    if labels:
        for x, y, s in ((1, 0, "1"), (-1, 0, "−1"), (0, 1, "1"), (0, -1, "−1")):
            ax.annotate(s, (x, y), xytext=(5, 5), textcoords="offset points", color=DIM, fontsize=11)
    return ax

def arrow(ax, angle, color=CORAL, lw=3.2, r=1.0, label=None, start=(0.0, 0.0)):
    """The turning arrow: length r, at `angle` radians, from `start`. Returns its tip."""
    x, y = start[0] + r * np.cos(angle), start[1] + r * np.sin(angle)
    ax.annotate("", xy=(x, y), xytext=start, arrowprops=dict(arrowstyle="-|>", color=color, lw=lw, mutation_scale=18))
    if label: ax.annotate(label, (x, y), xytext=(8, 6), textcoords="offset points", color=color)
    return x, y

def turn_arc(ax, angle, r=0.28, color=DIM):
    """A small arc from 0 to `angle`, showing which way the arrow turned."""
    t = np.linspace(0, angle, 100)
    ax.plot(r * np.cos(t), r * np.sin(t), color=color, lw=1.4)

def pi_ticks(ax, lo=-2 * np.pi, hi=2 * np.pi, step=np.pi / 2):
    """x ticks at multiples of π/2, labelled with π."""
    names = {0: "0", 1: "π/2", 2: "π", 3: "3π/2", 4: "2π", 5: "5π/2", 6: "3π",
             -1: "−π/2", -2: "−π", -3: "−3π/2", -4: "−2π", -5: "−5π/2", -6: "−3π"}
    ks = range(int(np.ceil(lo / step)), int(np.floor(hi / step)) + 1)
    ax.set_xticks([k * step for k in ks]); ax.set_xticklabels([names.get(k, f"{k}π/2") for k in ks])

def gaps(y, big=8.0):
    """Break a curve where it shoots off to infinity, so no false vertical lines are drawn."""
    y = np.array(y, dtype=float); y[np.abs(y) > big] = np.nan
    return y

def both_sides(ax, x, left, right, title, ylim=None, big=8.0):
    """An identity as two graphs: the left side thick, the right side dashed on top. If they are the same function,
    the dashes sit exactly on the thick line."""
    ax.plot(x, gaps(left, big), color=TEAL, lw=6, alpha=0.55, label="left side")
    ax.plot(x, gaps(right, big), color=CORAL, lw=1.8, ls="--", label="right side")
    ax.set_title(title, fontsize=13)
    if ylim: ax.set_ylim(*ylim)
    pi_ticks(ax, x.min(), x.max())

def check_identity(left, right, two=False, n=6, lo=-2 * np.pi, hi=2 * np.pi, ok=None, seed=0):
    """Check an identity at random angles: prints a small table and the largest difference (relative), which should
    be about 10⁻¹⁵, far below the 10⁻¹² we ask for. `ok` keeps away from angles where a side is undefined."""
    rng = np.random.default_rng(seed)
    x, y = rng.uniform(lo, hi, 400), rng.uniform(lo, hi, 400)
    keep = ok(x, y) if (ok and two) else (ok(x) if ok else np.ones_like(x, dtype=bool))
    x, y = x[keep][:n], y[keep][:n]
    L, R = (left(x, y), right(x, y)) if two else (left(x), right(x))
    head = f"{'x':>9}" + (f"{'y':>9}" if two else "") + f"{'left side':>16}{'right side':>16}"
    print(head)
    for i in range(len(x)):
        print(f"{x[i]:9.4f}" + (f"{y[i]:9.4f}" if two else "") + f"{L[i]:16.10f}{R[i]:16.10f}")
    gap = float(np.max(np.abs(L - R) / np.maximum(1.0, np.abs(R))))
    print(f"largest difference: {gap:.1e}")
    assert gap < 1e-12, gap
    return gap

away_from_sin0 = lambda x: np.abs(np.sin(x)) > 0.05    # keeps cot and cosec away from where they break
away_from_cos0 = lambda x: np.abs(np.cos(x)) > 0.05    # keeps tan and sec away from where they break

## 1 · Turn twice

Turn by x, then by x again. That's the sum formula with y = x:

* **sin 2x = 2 sin x cos x**
* **cos 2x = cos² x − sin² x**

Drag x and compare the arrow at 2x with what the formulas predict from the arrow at x.

In [ ]:
def twice(x_deg=35.0):
    x = math.radians(x_deg)
    s, c = math.sin(x), math.cos(x)
    fig, ax = plt.subplots(figsize=(6.6, 6.6))
    circle_axes(ax)
    arrow(ax, x, color=TEAL, label="x")
    tip = arrow(ax, 2 * x, label="2x")
    ax.plot([0, c * c - s * s], [2 * s * c] * 2, color=SAGE, ls=":"); ax.plot([c * c - s * s] * 2, [0, 2 * s * c], color=SAGE, ls=":")
    ax.plot([c * c - s * s], [2 * s * c], "o", ms=14, mfc="none", mec=SAGE, mew=2)
    ax.set_title(f"2 sin x cos x = {2 * s * c:+.4f},  sin 2x = {math.sin(2 * x):+.4f}\n"
                 f"cos² x − sin² x = {c * c - s * s:+.4f},  cos 2x = {math.cos(2 * x):+.4f}", fontsize=12)
    plt.show()

interact(twice, x_deg=FloatSlider(value=35, min=-180, max=180, step=5, description="x (°)"));

**Notice.** The sage ring, placed by the formulas, always sits on the tip of the arrow at 2x.

## 2 · Four faces of cos 2x

Use sin² x + cos² x = 1 to swap one square for the other, and cos 2x has four forms. NCERT lists them all:

cos 2x = cos² x − sin² x = 2 cos² x − 1 = 1 − 2 sin² x = (1 − tan² x) ÷ (1 + tan² x).

They are four ways of writing one number. The bars show all four for your angle.

In [ ]:
def four_faces(x_deg=35.0):
    x = math.radians(x_deg); s, c, t = math.sin(x), math.cos(x), math.tan(x)
    forms = {"cos² − sin²": c * c - s * s, "2cos² − 1": 2 * c * c - 1, "1 − 2sin²": 1 - 2 * s * s,
             "(1 − tan²)/(1 + tan²)": (1 - t * t) / (1 + t * t)}
    fig, ax = plt.subplots(figsize=(9.5, 3.8))
    ax.barh(list(forms), list(forms.values()), color=[TEAL, SAGE, LILAC, CORAL], height=0.55)
    ax.axvline(math.cos(2 * x), color=CHALK, lw=1.5, ls="--")
    ax.set_xlim(-1.1, 1.1); ax.set_title(f"cos 2x at x = {x_deg:.0f}°: {math.cos(2 * x):+.4f}, four ways")
    plt.tight_layout(); plt.show()

interact(four_faces, x_deg=FloatSlider(value=35, min=-85, max=85, step=5, description="x (°)"));

## 3 · Turn three times

Turn by 2x, then by x once more, and use the formulas again. Replacing every cos² by 1 − sin² gives:

* **sin 3x = 3 sin x − 4 sin³ x**
* **cos 3x = 4 cos³ x − 3 cos x**
* **tan 3x = (3 tan x − tan³ x) ÷ (1 − 3 tan² x)**

Each is a polynomial in one projection. That's the key to Viète's trick.

In [ ]:
def thrice(x_deg=25.0):
    x = math.radians(x_deg); s, c = math.sin(x), math.cos(x)
    fig, ax = plt.subplots(figsize=(6.6, 6.6))
    circle_axes(ax)
    for k, col in ((1, TEAL), (2, SAGE), (3, CORAL)): arrow(ax, k * x, color=col, label=f"{k}x")
    ax.set_title(f"3 sin x − 4 sin³ x = {3 * s - 4 * s ** 3:+.4f},  sin 3x = {math.sin(3 * x):+.4f}\n"
                 f"4 cos³ x − 3 cos x = {4 * c ** 3 - 3 * c:+.4f},  cos 3x = {math.cos(3 * x):+.4f}", fontsize=12)
    plt.show()

interact(thrice, x_deg=FloatSlider(value=25, min=-120, max=120, step=5, description="x (°)"));

## 4 · Any number of turns: cos nx is a polynomial in cos x

Turning n + 1 times and n − 1 times are related by the sum formulas:
cos((n + 1)x) = 2 cos x cos nx − cos((n − 1)x). Start from cos 0x = 1 and cos 1x = cos x, and this builds a
polynomial in c = cos x for every n. Pick n and see it.

In [ ]:
def cos_poly(n):
    """Coefficients (lowest power first) of the polynomial T_n with cos(nx) = T_n(cos x)."""
    a, b = [1], [0, 1]
    if n == 0: return a
    for _ in range(n - 1):
        c = [0] + [2 * v for v in b]
        a2 = a + [0] * (len(c) - len(a))
        a, b = b, [ci - ai for ci, ai in zip(c, a2)]
    return b

SUP = str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹")

def show_poly(n=3):
    p = cos_poly(n)
    terms = [f"{'+' if v > 0 else '−'} {abs(v) if (abs(v) != 1 or k == 0) else ''}{'c' if k else ''}{str(k).translate(SUP) if k > 1 else ''}"
             for k, v in enumerate(p) if v][::-1]
    expr = " ".join(terms).lstrip("+ ")
    x = np.linspace(-np.pi, np.pi, 800)
    fig, ax = plt.subplots(figsize=(10, 3.8))
    both_sides(ax, x, np.cos(n * x), np.polynomial.polynomial.polyval(np.cos(x), p), f"cos {n}x = {expr}"[:95])
    ax.legend(loc="lower right"); plt.show()

interact(show_poly, n=IntSlider(value=3, min=1, max=12, description="n"));

## 5 · Half angles

Read cos 2x = 2 cos² x − 1 = 1 − 2 sin² x backwards, with x/2 in place of x:
**cos² (x/2) = (1 + cos x) ÷ 2** and **sin² (x/2) = (1 − cos x) ÷ 2**. The quarter of the circle that x/2 lies in
decides the signs of the square roots.

## 6 · Van Roomen's challenge, solved

Van Roomen's left side is a polynomial of degree 45 that starts 45x − 3795x³ + 95634x⁵ − … and ends with x⁴⁵.
With x = 2 sin θ it is exactly **2 sin 45θ**: forty-five turns, written in one projection. His right side,
√(7/4 − √(5/16) − √(15/8 − √(45/64))), is exactly 2 sin 12°. So the equation says sin 45θ = sin 12°.

In [ ]:
from fractions import Fraction

def dickson(n):
    """Coefficients of D_n with 2 cos(nφ) = D_n(2 cos φ); for odd n it also gives 2 sin(nθ) from 2 sin θ."""
    a, b = [2], [0, 1]
    for _ in range(n - 1):
        c = [0] + b
        a2 = a + [0] * (len(c) - len(a))
        a, b = b, [ci - ai for ci, ai in zip(c, a2)]
    return b

P = dickson(45)
print("degree", len(P) - 1, "; first terms:", [(k, P[k]) for k in (1, 3, 5, 7)], "; last:", P[45])
assert (P[1], P[3], P[5], P[45]) == (45, -3795, 95634, 1)
RHS = math.sqrt(7 / 4 - math.sqrt(5 / 16) - math.sqrt(15 / 8 - math.sqrt(45 / 64)))
print(f"right side = {RHS:.15f},  2 sin 12° = {2 * math.sin(math.radians(12)):.15f}")
assert abs(RHS - 2 * math.sin(math.radians(12))) < 1e-14

def P_exact(x):
    """Evaluate the polynomial exactly (its coefficients are huge, so ordinary floats would lose the answer)."""
    s, X = Fraction(0), Fraction(x)
    for c in reversed(P): s = s * X + c
    return float(s)

**The 23 positive roots.** If 45θ = 12° + 360° k, then sin 45θ = sin 12°. So θ = (12° + 360° k) ÷ 45, and each root is
x = 2 sin θ. For k = 0 to 22, θ stays between 0° and 180°, so x is positive: 23 roots.

In [ ]:
roots = [2 * math.sin(math.radians((12 + 360 * k) / 45)) for k in range(23)]
worst = max(abs(P_exact(r) - RHS) for r in roots)
for k in (0, 1, 2, 11, 22): print(f"k = {k:2d}: x = {roots[k]:.12f}")
print(f"23 positive roots; largest |left − right| = {worst:.1e}")
assert worst < 1e-10 and len(set(round(r, 12) for r in roots)) == 23

In [ ]:
xs = np.linspace(-2, 2, 1200)
ys = np.array([P_exact(v) for v in xs])
fig, ax = plt.subplots(figsize=(11.5, 4.4))
ax.plot(xs, ys, color=TEAL, lw=1.6, label="van Roomen's left side")
ax.axhline(RHS, color=CORAL, lw=1.5, label="his right side = 2 sin 12°")
ax.plot(roots, [RHS] * 23, "o", color=CORAL, ms=6, label="Viète's 23 positive roots")
ax.set_ylim(-2.5, 2.5); ax.set_xlabel("x"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=11)
ax.set_title("a degree-45 polynomial is 45 turns in disguise"); plt.tight_layout(); plt.show()

**Notice.** The polynomial wiggles between −2 and 2, because it *is* 2 sin 45θ. Each crossing of the coral line is a
root. The 22 negative roots are the mirror images; Viète set them aside.

## 7 · The identity gallery

The class 11 list in full (NCERT identities 14–19), plus the half-angle forms.

In [ ]:
x = np.linspace(-2 * np.pi, 2 * np.pi, 1500)
t = np.tan
gallery = [("sin 2x = 2 sin x cos x", lambda v: np.sin(2 * v), lambda v: 2 * np.sin(v) * np.cos(v), None),
           ("sin 2x = 2 tan x ÷ (1 + tan² x)", lambda v: np.sin(2 * v), lambda v: 2 * t(v) / (1 + t(v) ** 2), away_from_cos0),
           ("cos 2x = cos² x − sin² x", lambda v: np.cos(2 * v), lambda v: np.cos(v) ** 2 - np.sin(v) ** 2, None),
           ("cos 2x = 2 cos² x − 1", lambda v: np.cos(2 * v), lambda v: 2 * np.cos(v) ** 2 - 1, None),
           ("cos 2x = 1 − 2 sin² x", lambda v: np.cos(2 * v), lambda v: 1 - 2 * np.sin(v) ** 2, None),
           ("cos 2x = (1 − tan² x) ÷ (1 + tan² x)", lambda v: np.cos(2 * v), lambda v: (1 - t(v) ** 2) / (1 + t(v) ** 2), away_from_cos0),
           ("tan 2x = 2 tan x ÷ (1 − tan² x)", lambda v: t(2 * v), lambda v: 2 * t(v) / (1 - t(v) ** 2),
            lambda v: (np.abs(np.cos(v)) > 0.1) & (np.abs(np.cos(2 * v)) > 0.1)),
           ("sin 3x = 3 sin x − 4 sin³ x", lambda v: np.sin(3 * v), lambda v: 3 * np.sin(v) - 4 * np.sin(v) ** 3, None),
           ("cos 3x = 4 cos³ x − 3 cos x", lambda v: np.cos(3 * v), lambda v: 4 * np.cos(v) ** 3 - 3 * np.cos(v), None),
           ("tan 3x = (3 tan x − tan³ x) ÷ (1 − 3 tan² x)", lambda v: t(3 * v), lambda v: (3 * t(v) - t(v) ** 3) / (1 - 3 * t(v) ** 2),
            lambda v: (np.abs(np.cos(v)) > 0.1) & (np.abs(np.cos(3 * v)) > 0.1)),
           ("cos²(x/2) = (1 + cos x) ÷ 2", lambda v: np.cos(v / 2) ** 2, lambda v: (1 + np.cos(v)) / 2, None),
           ("sin²(x/2) = (1 − cos x) ÷ 2", lambda v: np.sin(v / 2) ** 2, lambda v: (1 - np.cos(v)) / 2, None)]
fig, axs = plt.subplots(6, 2, figsize=(12, 16))
for ax, (name, L, R, _) in zip(axs.flat, gallery):
    with np.errstate(divide="ignore", invalid="ignore"):
        both_sides(ax, x, L(x), R(x), name, ylim=(-4, 4))
plt.tight_layout(); plt.show()
for name, L, R, ok in gallery:
    print(name); check_identity(L, R, n=3, ok=ok)

## 8 · On paper: the CBSE way

**Example 1.** If tan x = 3/4 and π < x < 3π/2, find sin(x/2), cos(x/2) and tan(x/2).

1. In the third quarter, cos x is negative: sec² x = 1 + 9/16 = 25/16, so cos x = −4/5.
2. π/2 < x/2 < 3π/4, the second quarter: sin(x/2) is positive and cos(x/2) is negative.
3. sin²(x/2) = (1 − cos x) ÷ 2 = 9/10, so **sin(x/2) = 3/√10**. cos²(x/2) = (1 + cos x) ÷ 2 = 1/10, so
   **cos(x/2) = −1/√10**. Then **tan(x/2) = −3**.

In [ ]:
x1 = math.pi + math.atan(3 / 4)
print(f"sin(x/2) = {math.sin(x1 / 2):.6f} (3/√10 = {3 / math.sqrt(10):.6f}),  cos(x/2) = {math.cos(x1 / 2):.6f},  "
      f"tan(x/2) = {math.tan(x1 / 2):.6f}")

**Example 2.** If cos x = 1/3, find cos 3x.

1. cos 3x = 4 cos³ x − 3 cos x.
2. = 4 × 1/27 − 3 × 1/3 = 4/27 − 1.
3. **cos 3x = −23/27.**

In [ ]:
print(f"cos 3x = {math.cos(3 * math.acos(1 / 3)):.6f},  −23/27 = {-23 / 27:.6f}")

**Try it.**
1. sin x = 3/5 and x is in the first quarter. Find sin 2x.
2. tan x = 1/2. Find cos 2x.

<details><summary>Answers</summary>

1. cos x = 4/5, so sin 2x = 2 × 3/5 × 4/5 = 24/25.
2. cos 2x = (1 − 1/4) ÷ (1 + 1/4) = 3/5.

</details>

## Check yourself
1. What is sin 60°, using sin 2x with x = 30°?
2. Write cos 2x using sin x only.
3. How many positive roots did Viète find for van Roomen's equation?

<details><summary>Answers</summary>

1. 2 sin 30° cos 30° = 2 × 1/2 × √3/2 = √3/2.
2. cos 2x = 1 − 2 sin² x.
3. 23.

</details>

## Sources
MacTutor History of Mathematics (St Andrews): "François Viète"; "Adriaan van Roomen". W. Van Assche, Journal of
Approximation Theory 279 (2022), on van Roomen's equation (arXiv 2203.10955). University of California, Berkeley,
notes on Viète. NCERT class 11, chapter 3 (identities 14–19). CBSE Mathematics syllabus 2026–27. All retrieved
2026-10-21.